# Using an N-body snapshot

This example uses a single snapshot to draw a 3d map.

In [ ]:
# basic imports and logging
import numpy as np
import os
import matplotlib.pyplot as plt
import logging
from pathlib import Path

logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

In [ ]:
# importing beorn, assuming it has already been installed
import beorn

In [ ]:
# setup storage paths
CACHE_ROOT = Path(os.getenv("SCRATCH", default = "."))


project_storage = os.getenv("PROJECT")
if project_storage is None:
    FILE_ROOT = Path(".")
else:
    # in that case wer are on a HPC cluster
    FILE_ROOT = Path(project_storage) / "input" / "200Mpc" / "CDM_200Mpc_2048"


cache_handler = beorn.io.Handler(CACHE_ROOT / ".beorn_cache")
output_handler = beorn.io.Handler(FILE_ROOT / "output")

### Parameter setup
This includes cosmic models and the models of sources of radiation.

For more details on parameter definitions, see [arXiv/2305.15466](https://arxiv.org/abs/2305.15466).

In [ ]:
parameters = beorn.structs.Parameters()

## Simulation parameters, i.e. the allowed "dynamic range" of the simulation
parameters.simulation.cores = 60

parameters.simulation.halo_mass_bin_min = 1e7
parameters.simulation.halo_mass_bin_max = 1e15
parameters.simulation.halo_mass_bin_n = 200
parameters.simulation.halo_mass_accretion_alpha = np.array([0.785, 0.795])


## Source parameters
# emission
parameters.source.Nion = 300
parameters.source.energy_cutoff_min_xray = 500
parameters.source.energy_cutoff_max_xray = 2000
parameters.source.energy_min_sed_xray = 200
parameters.source.energy_max_sed_xray = 2000
parameters.source.alS_xray = 1.5
parameters.source.xray_normalisation = 3.4e40 * 0.1

# fesc
parameters.source.f0_esc = 0.2
parameters.source.pl_esc = 0.5

# fstar
parameters.source.f_st = 0.05
parameters.source.g1 = 0.49
parameters.source.g2 = -0.61
parameters.source.g3 = 4
parameters.source.g4 = -4
parameters.source.Mp = 1.6e11 * parameters.cosmology.h
parameters.source.Mt = 1e9

# Minimum star forming halo
parameters.source.halo_mass_min = 1e8


# Adjusted to the PKDGRAV data
parameters.solver.redshifts = [6, 25]
parameters.simulation.file_root = FILE_ROOT

In [ ]:
# specify the input format by using the appropriate loader (PKDGrav-compatible in this case)
loader = beorn.load_input_data.PKDGravLoader(parameters)

### Inspect the available snapshots

In [ ]:
catalog = loader.load_halo_catalog(20)

halo_count = catalog.size
mesh = catalog.to_mesh()
mesh_flattened = np.sum(mesh, axis=0)


plt.figure()
plt.imshow(mesh_flattened, origin="lower", interpolation="none")
plt.show()

### Compute and plot radiation profiles (ly-al, xHII, Tk)

In [ ]:
solver = beorn.precomputation.RadiationProfileSolver(parameters, loader.redshifts)
profiles = solver.get_or_compute_profiles(cache_handler)

In [ ]:
mass_index = parameters.simulation.halo_mass_bin_n // 2

beorn.plotting.plot_1D_profiles(
    parameters,
    profiles,
    mass_index,
    redshifts = [13,10,8],
    alphas = [0.79]
)

### Painting
Using the radiation profiles and the halo catalogs we can generate full 3d maps of the ionization

In [ ]:
output_handler = beorn.io.Handler(CACHE_ROOT / ".pkdgrav_output")

p = beorn.painting.Painter(
    parameters,
    cache_handler = cache_handler,
    output_handler = output_handler,
    loader = loader
)

grids = p.paint_full(profiles)

In [ ]:
# TODO